# BigQuery: Global Queries Demo (Preview)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/bq_global_queries_demo.ipynb)

This notebook demonstrates the **[Global Queries (Preview)](https://docs.cloud.google.com/bigquery/docs/global-queries)** feature in BigQuery, which allows you to reference data stored in more than one region in a single query. [Release Details](https://docs.cloud.google.com/bigquery/docs/release-notes#February_17_2026)

## Use Case
A global enterprise has sales data in `us-central1` and inventory data in `europe-west1`. Previously, they had to export/import data to join these tables. Now, they can join them directly.

### Requirements
- BigQuery API enabled.
- **Permissions**: `bigquery.jobs.createGlobalQuery` (included in `BigQuery Admin`).
- **Project setting**: Enable `enable_global_queries_execution` and `enable_global_queries_data_access`  for specific reagion.
- Datasets in at least two different regions (e.g., `us-central1` and `europe-west1`).

In [ ]:
# 1. Setup and Authentication
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

from google.cloud import bigquery
project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
client = bigquery.Client(project=project_id)

### 2. [MANDATORY] Enable Global Queries for your Project

By default, cross-region joins are disabled. You must run these DDL statements to opt-in for each specific region pair.

In [ ]:
# Enable Execution in us-central1
setup_query_exec = f"""
SET @@location = 'us-central1';
ALTER PROJECT `{project_id}` SET OPTIONS (
  `region-us-central1.enable_global_queries_execution` = true
);
"""

# Enable Data Access in europe-west1
setup_query_access = f"""
SET @@location = 'europe-west1';
ALTER PROJECT `{project_id}` SET OPTIONS (
  `region-europe-west1.enable_global_queries_data_access` = true
);
"""

print("Enabling Global Query features...")
try:
    client.query(setup_query_exec).result()
    client.query(setup_query_access).result()
    print("Success: Project configured for Global Queries.")
except Exception as e:
    print(f"Warning: Could not configure project. Error: {str(e)}")

### 3. [PREREQUISITES] Create Datasets, Tables, and Dummy Data

This block sets up the environment in `us-central1` and `europe-west1`.

In [ ]:
def setup_demo_data():
    us_dataset_id = f"{project_id}.us_sales"
    eu_dataset_id = f"{project_id}.eu_inventory"

    # Create US Dataset (us-central1)
    us_dataset = bigquery.Dataset(us_dataset_id)
    us_dataset.location = "us-central1"
    client.create_dataset(us_dataset, exists_ok=True)

    # Create EU Dataset (europe-west1)
    eu_dataset = bigquery.Dataset(eu_dataset_id)
    eu_dataset.location = "europe-west1"
    client.create_dataset(eu_dataset, exists_ok=True)

    # Create US Sales Table
    sales_table_id = f"{us_dataset_id}.sales_data"
    sales_schema = [
        bigquery.SchemaField("product_id", "STRING"),
        bigquery.SchemaField("units_sold", "INTEGER"),
        bigquery.SchemaField("sale_date", "DATE"),
    ]
    client.create_table(bigquery.Table(sales_table_id, schema=sales_schema), exists_ok=True)
    client.insert_rows_json(sales_table_id, [
        {"product_id": "PROD_998", "units_sold": 450, "sale_date": "2026-03-09"},
        {"product_id": "PROD_442", "units_sold": 120, "sale_date": "2026-03-09"}
    ])

    # Create EU Inventory Table
    inventory_table_id = f"{eu_dataset_id}.inventory_data"
    inventory_schema = [
        bigquery.SchemaField("product_id", "STRING"),
        bigquery.SchemaField("stock_level", "INTEGER"),
        bigquery.SchemaField("warehouse_location", "STRING"),
    ]
    client.create_table(bigquery.Table(inventory_table_id, schema=inventory_schema), exists_ok=True)
    client.insert_rows_json(inventory_table_id, [
        {"product_id": "PROD_998", "stock_level": 1200, "warehouse_location": "Frankfurt-DE"},
        {"product_id": "PROD_442", "stock_level": 300, "warehouse_location": "London-UK"}
    ])
    print("Demo data prepared successfully.")

setup_demo_data()

### 4. Define and Execute the Global Query

We use `SET @@location = 'us-central1'` to run the query in the US primary region.

In [ ]:
query = f"""
SET @@location = 'us-central1';
SELECT
    sales.product_id,
    sales.units_sold,
    inventory.stock_level,
    inventory.warehouse_location
FROM
    `{project_id}.us_sales.sales_data` AS sales
INNER JOIN
    `{project_id}.eu_inventory.inventory_data` AS inventory
ON
    sales.product_id = inventory.product_id
WHERE
    sales.sale_date = '2026-03-09';
"""

print("Executing Global Query (Primary: us-central1)...")
query_job = client.query(query)
results = query_job.to_dataframe()
results.head()

### 5. Things to remember or know
- **Region Specificity**: Use exact region names like `us-central1` rather than multi-region names like `us` for precise control.
- **Location Variable**: In scripts, always set `@@location` before running `ALTER PROJECT` to avoid session errors.
- **Zero Data Movement**: BigQuery handles the cross-region orchestration natively.